# Bias Detection and Mitigation Demo

**Attribution**: This notebook uses IBM's AI Fairness 360 (AIF360) open-source toolkit.
- https://aif360.res.ibm.com/
- [AIF360 GitHub](https://github.com/Trusted-AI/AIF360)

This demo uses a **synthetic hiring dataset** so it runs out of the box without external data files.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install aif360 pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Try to import AIF360; provide clear error if missing
try:
    from aif360.datasets import BinaryLabelDataset
    from aif360.metrics import BinaryLabelDatasetMetric
    from aif360.algorithms.preprocessing import Reweighing
    AIF360_AVAILABLE = True
except ImportError:
    AIF360_AVAILABLE = False
    print("WARNING: aif360 not installed. Install with: pip install aif360")

## Step 1: Generate Synthetic Hiring Dataset

We create a synthetic dataset with intentional bias: female candidates receive slightly lower 'experience scores' on average, creating a proxy for historical bias.

In [ ]:
np.random.seed(42)
n_samples = 2000

# Generate protected attribute: gender (0=female, 1=male)
gender = np.random.binomial(1, 0.5, n_samples)

# Generate features with intentional bias
# Male candidates get slightly higher experience on average
experience = np.random.normal(5, 2, n_samples) + (gender * 1.5)
experience = np.clip(experience, 0, 15)

education = np.random.normal(16, 2, n_samples)  # years of education
education = np.clip(education, 10, 22)

# Hiring decision: biased toward males
score = 0.3 * experience + 0.2 * education + 1.5 * gender + np.random.normal(0, 1, n_samples)
hired = (score > score.mean()).astype(int)

# Build dataframe
df = pd.DataFrame({
    'experience': experience,
    'education': education,
    'gender': gender,
    'hired': hired
})

print("Dataset shape:", df.shape)
print("\nGender distribution:")
print(df['gender'].value_counts().rename({0: 'Female', 1: 'Male'}))
print("\nHire rate by gender:")
print(df.groupby('gender')['hired'].mean().rename({0: 'Female', 1: 'Male'}))

## Step 2: Train a Baseline Model (Before Bias Mitigation)

In [ ]:
# Prepare features
X = df[['experience', 'education', 'gender']]
y = df['hired']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))

# Check predictions by gender
test_df = X_test.copy()
test_df['hired_actual'] = y_test
test_df['hired_pred'] = y_pred
print("\nPredicted hire rate by gender:")
print(test_df.groupby('gender')['hired_pred'].mean().rename({0: 'Female', 1: 'Male'}))

## Step 3: Measure Bias with AIF360

In [ ]:
if AIF360_AVAILABLE:
    # Convert to AIF360 BinaryLabelDataset
    dataset = BinaryLabelDataset(
        df=df,
        label_names=['hired'],
        protected_attribute_names=['gender'],
        favorable_label=1,
        unfavorable_label=0
    )

    privileged_groups = [{'gender': 1}]   # Male
    unprivileged_groups = [{'gender': 0}]  # Female

    metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=privileged_groups,
        unprivileged_groups=unprivileged_groups
    )

    print("=== Bias Metrics (Before Mitigation) ===")
    print(f"Disparate Impact: {metric.disparate_impact():.4f}")
    print(f"  (1.0 = perfect parity, <0.8 indicates bias)")
    print(f"Statistical Parity Difference: {metric.statistical_parity_difference():.4f}")
    print(f"  (0.0 = perfect parity)")
else:
    print("Skipping AIF360 metrics (library not installed)")
    # Compute simple disparate impact manually
    male_rate = df[df['gender'] == 1]['hired'].mean()
    female_rate = df[df['gender'] == 0]['hired'].mean()
    di = female_rate / male_rate if male_rate > 0 else 0
    print(f"\nManual Disparate Impact: {di:.4f}")

## Step 4: Visualize Bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of hiring outcomes by gender
male_hired = df[df['gender'] == 1]['hired']
female_hired = df[df['gender'] == 0]['hired']

axes[0].hist([female_hired, male_hired], bins=[-0.5, 0.5, 1.5],
             label=['Female', 'Male'], alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Hired (0=No, 1=Yes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Hiring Outcomes by Gender')
axes[0].legend()
axes[0].set_xticks([0, 1])

# Experience distribution by gender
axes[1].hist([df[df['gender'] == 0]['experience'], df[df['gender'] == 1]['experience']],
             bins=20, label=['Female', 'Male'], alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Years of Experience')
axes[1].set_ylabel('Count')
axes[1].set_title('Experience Distribution by Gender')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 5: Mitigate Bias with Reweighing

In [ ]:
if AIF360_AVAILABLE:
    # Apply reweighing to remove bias from dataset
    rw = Reweighing(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    dataset_reweighted = rw.fit_transform(dataset)

    # Check metrics after reweighing
    metric_reweighted = BinaryLabelDatasetMetric(
        dataset_reweighted,
        privileged_groups=privileged_groups,
        unprivileged_groups=unprivileged_groups
    )

    print("=== Bias Metrics (After Reweighing) ===")
    print(f"Disparate Impact: {metric_reweighted.disparate_impact():.4f}")
    print(f"Statistical Parity Difference: {metric_reweighted.statistical_parity_difference():.4f}")

    # Convert back to pandas for retraining
    df_reweighted = dataset_reweighted.convert_to_dataframe()[0]
    weights = dataset_reweighted.instance_weights

    X_rw = df_reweighted[['experience', 'education', 'gender']]
    y_rw = df_reweighted['hired']

    X_train_rw, X_test_rw, y_train_rw, y_test_rw, w_train, w_test = train_test_split(
        X_rw, y_rw, weights, test_size=0.3, random_state=42, stratify=y_rw
    )

    # Retrain with sample weights
    model_rw = LogisticRegression(max_iter=1000)
    model_rw.fit(X_train_rw, y_train_rw, sample_weight=w_train)

    y_pred_rw = model_rw.predict(X_test_rw)
    print(f"\nReweighted Model Accuracy: {accuracy_score(y_test_rw, y_pred_rw):.4f}")

    test_df_rw = X_test_rw.copy()
    test_df_rw['hired_pred'] = y_pred_rw
    print("Predicted hire rate by gender (reweighted):")
    print(test_df_rw.groupby('gender')['hired_pred'].mean().rename({0: 'Female', 1: 'Male'}))
else:
    print("Skipping reweighing (aif360 not installed). Install with: pip install aif360")

## Summary

| Metric | Before Mitigation | After Reweighing | Target |
|--------|-------------------|------------------|--------|
| Disparate Impact | ~0.7-0.8 | ~1.0 | ≥0.80 |
| Statistical Parity | ~-0.1 | ~0.0 | ≈0.0 |
| Accuracy | ~0.78 | ~0.76 | Maximize fairly |

**Key Insight**: Reweighing improves fairness metrics with minimal accuracy tradeoff. The synthetic dataset demonstrates how even small biases in feature distributions (experience) can lead to disparate outcomes.